# NUTDTS 816 Time Series Analysis
## L15 Time series as supervised learning; gradient boosting

Lab notebook for Chapter 8 of the lecture notes. Run the setup cell first. Every code cell reproduces an example from the notes; the exercises at the end are from the chapter's self-check list.

**Instructor:** Dr Tolulope Adesina · NUTM MSc Data Science · 2026

In [ ]:
# ---- Setup: run once per Colab session ----
# 1. Install the course libraries (about two minutes the first time)
!pip install -q statsmodels pmdarima statsforecast neuralforecast lightgbm arch plotly

# 2. Fetch the course data module and data snapshots from the course repository.
#    Replace REPO with your fork or the official course repository URL.
REPO = "https://raw.githubusercontent.com/<your-github-user>/nutdts816/main"
import urllib.request, os
os.makedirs("data", exist_ok=True)
urllib.request.urlretrieve(f"{REPO}/src/tsdata.py", "tsdata.py")
DATA_FILES = ["nigeria_cpi", "nigeria_fx", "nigeria_grid", "nigeria_malaria", "nigeria_rainfall", "bonny_light", "daily_demand",
              "airpassengers", "a10", "h02", "ausbeer", "elecequip", "usmelec", "goog", "nile", "austourists", "oil", "dax", "uschange", "elecdemand"]
for f in DATA_FILES:
    urllib.request.urlretrieve(f"{REPO}/data/{f}.csv", f"data/{f}.csv")

import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (9, 3.6), "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})
import tsdata
print("Setup complete.")

### 8.2 Feature engineering

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import tsdata
dem = tsdata.daily_demand()     # simulated daily demand (GWh), 2022-2025, weekly + annual seasonality, holidays, trend
NG_HOLIDAYS = [f'{y}-{md}' for y in range(2022, 2027) for md in ['01-01', '05-01', '06-12', '10-01', '12-25', '12-26']]

def make_features(y, horizon, lags=(1, 2, 3, 7, 14, 21, 28, 364), windows=(7, 28, 91), keep_unlabelled=False):
    """Feature table for a DIRECT h-step forecaster: row t uses information up to t, target is y[t + horizon].
    keep_unlabelled=True keeps the final rows whose target is not yet observed (used to build the forecast row)."""
    df = pd.DataFrame(index=y.index)
    for k in lags: df[f'lag{k}'] = y.shift(k)
    for w in windows:
        df[f'rmean{w}'] = y.shift(1).rolling(w).mean(); df[f'rstd{w}'] = y.shift(1).rolling(w).std()
    df['diff7'] = y.shift(1) - y.shift(8)
    tgt_idx = y.index + pd.Timedelta(days=horizon)         # calendar features describe the TARGET date (known in advance)
    df['dow'] = tgt_idx.dayofweek; df['month'] = tgt_idx.month; df['doy_sin'] = np.sin(2 * np.pi * tgt_idx.dayofyear / 365.25); df['doy_cos'] = np.cos(2 * np.pi * tgt_idx.dayofyear / 365.25)
    df['holiday'] = tgt_idx.strftime('%Y-%m-%d').isin(NG_HOLIDAYS).astype(int)
    df['target'] = y.shift(-horizon)
    feats = [c for c in df.columns if c != 'target']
    return df.dropna(subset=feats) if keep_unlabelled else df.dropna()

F7 = make_features(dem, horizon=7)
print(F7.shape); print(F7.iloc[[0, -1], :8].round(2).to_string())

### 8.4 Gradient boosting for forecasting

In [ ]:
import lightgbm as lgb
# A trending series: can a tree model follow it?
rng = np.random.default_rng(7); t = np.arange(400); y_tr = pd.Series(50 + 0.2 * t + 5 * np.sin(2 * np.pi * t / 30) + rng.normal(0, 1.5, 400))
X = pd.DataFrame({'lag1': y_tr.shift(1), 'lag2': y_tr.shift(2), 'lag30': y_tr.shift(30), 'phase': np.sin(2 * np.pi * t / 30)}).dropna(); yy = y_tr.loc[X.index]
tr_idx = X.index < 300
m_level = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=15, verbose=-1).fit(X[tr_idx], yy[tr_idx])
m_diff  = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=15, verbose=-1).fit(X[tr_idx], (yy - X['lag1'])[tr_idx])
fig, ax = plt.subplots(figsize=(9, 3.2)); y_tr.plot(ax=ax, lw=1, label='series (trend + cycle)')
pd.Series(m_level.predict(X[~tr_idx]), index=X.index[~tr_idx]).plot(ax=ax, lw=2, color='#A0302A', label='LightGBM on the level: flattens at the training maximum')
pd.Series(X['lag1'][~tr_idx] + m_diff.predict(X[~tr_idx]), index=X.index[~tr_idx]).plot(ax=ax, lw=2, color='#B8860B', label='LightGBM on the difference: follows the trend')
ax.axvline(300, color='#555555', lw=0.8); ax.legend(fontsize=8); ax.set_title('Why the target must be made stationary for tree models')
_caption = 'Trained on the first 300 points, a boosted-tree model of the level cannot predict above the highest value it has seen; the same model trained on the one-step change has no such limit.'

In [ ]:
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
H = 14
split = int(len(dem) * 0.8); origins = list(range(split, len(dem) - H, 21))    # an origin every three weeks
print(f'{len(origins)} origins from {dem.index[split].date()}, horizon {H} days')

def lgbm_direct_forecast(y_train, H, params=dict(n_estimators=400, learning_rate=0.03, num_leaves=15, min_child_samples=20, subsample=0.8, colsample_bytree=0.8, verbose=-1)):
    fc = []
    for h in range(1, H + 1):
        F = make_features(y_train, h); F['target_d'] = F['target'] - F['lag1']          # direct model of the CHANGE from the last observed value
        feats = [c for c in F.columns if c not in ('target', 'target_d')]
        m = lgb.LGBMRegressor(**params).fit(F[feats], F['target_d'])
        row = make_features(y_train, h, keep_unlabelled=True).loc[[y_train.index[-1]], feats]   # features at the origin, target h days ahead
        fc.append(y_train.iloc[-1] + m.predict(row)[0])
    return np.array(fc)

errs = {'LightGBM direct': [], 'ETS(A,Ad,A) weekly': [], 'Seasonal naive (7)': []}
for T in origins:
    ytr, act = dem.iloc[:T], dem.iloc[T:T + H].values
    errs['LightGBM direct'].append(act - lgbm_direct_forecast(ytr, H))
    ets = ETSModel(ytr, error='add', trend='add', damped_trend=True, seasonal='add', seasonal_periods=7, initialization_method='estimated').fit(disp=False)
    errs['ETS(A,Ad,A) weekly'].append(act - ets.forecast(H).values)
    errs['Seasonal naive (7)'].append(act - np.tile(ytr.iloc[-7:].values, 2)[:H])
q = np.mean(np.abs(dem.values[7:] - dem.values[:-7]))
mase = pd.DataFrame({k: np.mean(np.abs(np.array(v)), axis=0) / q for k, v in errs.items()}, index=[f'h={k}' for k in range(1, H + 1)])
print(mase.round(3).iloc[[0, 1, 2, 6, 13]].to_string()); print('\nAverage MASE over 14 horizons:'); print(mase.mean().round(3).to_string())

In [ ]:
ax = mase.plot(figsize=(8.5, 3.4), marker='o', ms=3, lw=1.5); ax.set_xlabel('horizon (days)'); ax.set_ylabel('MASE (scaled by weekly naive)'); ax.set_title('Daily demand: rolling-origin accuracy by horizon'); ax.legend(fontsize=8)
_caption = 'A split decision: the weekly ETS is more accurate for the first few days, the direct LightGBM forecaster from about a week out, and both beat the seasonal naive throughout.'

In [ ]:
# Which features matter? Gain-based importance from the 7-day-ahead model
F = make_features(dem.iloc[:split], 7); F['target_d'] = F['target'] - F['lag1']; feats = [c for c in F.columns if c not in ('target', 'target_d')]
m7 = lgb.LGBMRegressor(n_estimators=400, learning_rate=0.03, num_leaves=15, verbose=-1).fit(F[feats], F['target_d'])
imp = pd.Series(m7.booster_.feature_importance('gain'), index=feats).sort_values(ascending=False)
print((100 * imp / imp.sum()).round(1).head(10).to_string())

## Exercises

1. Write the leakage check for a feature table: for a given row, list every feature and the latest date its computation touches. Apply it to `make_features(dem, 14)` for the row indexed 2025-06-30.
2. Build a recursive one-step LightGBM forecaster for the daily demand series and compare it with the direct forecaster at horizons 1, 7 and 14 on the same rolling origin. Explain the pattern.
3. Add Fourier terms and holiday dummies to a dynamic harmonic regression (Chapter 5) for the daily series and include it in the rolling-origin comparison. How much of LightGBM's advantage disappears?
4. Train LightGBM quantile models ($\alpha = 0.1, 0.9$) for the 7-day horizon and report the empirical coverage of the resulting 80% interval on the rolling origin. Then build a conformal interval from horizon-7 backtest errors and compare.

In [ ]:
# Your work here
